# 📊 Section 4: Advanced Seasonal Hybrid Forecasting (v9 Plus)

**Mục tiêu**: Xây dựng hệ thống dự báo doanh thu tối ưu kết hợp giữa Hồ sơ mùa vụ (Seasonal Profile) và Học máy (Gradient Boosting).

**Chiến lược v9 Plus (Synchronized)**:
1. **Seasonal Profiling**: Trích xuất quy luật doanh thu hàng ngày dựa trên 3 năm gần nhất (2020-2022).
2. **LightGBM Hybrid**: Sử dụng mô hình boosting để bắt các biến động phi tuyến dựa trên các tính năng lịch mở rộng (Sin/Cos year & week).
3. **Dynamic Level Estimation**: Ước tính mức doanh thu cơ sở dựa trên xu hướng năm 2022 với slope được hiệu chỉnh.
4. **Weighted Ensemble**: Kết hợp hai phương pháp dựa trên trọng số MAE từ quá trình Backtesting 2022.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Paths
DATA_PATH = Path('../Data/processed_train.csv')
SUB_PATH = Path('../Data/sample_submission.csv')

# Load Data
df = pd.read_csv(DATA_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
sub = pd.read_csv(SUB_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

print(f'Training Data: {df.shape[0]} rows | {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Submission Template: {sub.shape[0]} rows | {sub["Date"].min().date()} to {sub["Date"].max().date()}')

## 1. Feature Engineering

Chúng tôi xây dựng các tính năng đặc trưng cho thị trường Việt Nam, bao gồm Fourier Seasonality và Tết Score.

In [ ]:
TET_DATES = pd.to_datetime([
    '2012-01-23','2013-02-10','2014-01-31','2015-02-19',
    '2016-02-08','2017-01-28','2018-02-16','2019-02-05',
    '2020-01-25','2021-02-12','2022-02-01','2023-01-22','2024-02-10',
])

def calendar_features(dates):
    d = pd.DataFrame({'Date': dates})
    dt = d['Date']
    d['month'] = dt.dt.month
    d['day'] = dt.dt.day
    d['dayofweek'] = dt.dt.dayofweek
    d['dayofyear'] = dt.dt.dayofyear
    d['quarter'] = dt.dt.quarter
    d['is_weekend'] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    d['is_month_end'] = dt.dt.is_month_end.astype(int)
    d['is_payday'] = dt.dt.day.isin([15, 30, 31]).astype(int)
    
    # Fourier Features (K=3)
    for k in [1, 2, 3]:
        d[f'sin_yr_{k}'] = np.sin(2 * np.pi * k * d['dayofyear'] / 365.25)
        d[f'cos_yr_{k}'] = np.cos(2 * np.pi * k * d['dayofyear'] / 365.25)
        d[f'sin_wk_{k}'] = np.sin(2 * np.pi * k * d['dayofweek'] / 7)
        d[f'cos_wk_{k}'] = np.cos(2 * np.pi * k * d['dayofweek'] / 7)

    # Tet Features
    d['days_to_tet'] = 999
    for tet in TET_DATES:
        diff = (dt - tet).dt.days
        mask = (diff >= -30) & (diff <= 45)
        d.loc[mask, 'days_to_tet'] = diff[mask]
    
    d['is_pre_tet'] = ((d['days_to_tet'] >= -30) & (d['days_to_tet'] < 0)).astype(int)
    d['is_tet_week'] = ((d['days_to_tet'] >= 0) & (d['days_to_tet'] <= 7)).astype(int)
    d['is_post_tet'] = ((d['days_to_tet'] > 7) & (d['days_to_tet'] <= 45)).astype(int)
    d['tet_score'] = np.where(d['days_to_tet'] == 999, 0, np.exp(-0.05 * np.abs(d['days_to_tet'])))
    
    # Holiday
    d['is_holiday'] = 0
    for m, dy in [(1,1),(4,30),(5,1),(9,2),(12,25)]:
        d.loc[(d['month']==m) & (d['day']==dy), 'is_holiday'] = 1
        
    return d.drop(columns=['Date'])

FEAT_COLS = calendar_features(df['Date']).columns.tolist()
print(f'Generated {len(FEAT_COLS)} features.')

## 2. Seasonal Profiling Logic

In [ ]:
def build_seasonal_profile(df, target, recent_years=3):
    df2 = df.copy()
    df2['year'] = df2['Date'].dt.year
    df2['month'] = df2['Date'].dt.month
    df2['dom'] = df2['Date'].dt.day
    
    max_year = df2['year'].max()
    recent = df2[df2['year'] >= max_year - recent_years + 1].copy()
    
    annual_mean = recent.groupby('year')[target].mean()
    recent['norm'] = recent[target] / recent['year'].map(annual_mean)
    
    profile = recent.groupby(['month', 'dom'])['norm'].mean().reset_index()
    profile.columns = ['month', 'dom', 'seasonal_ratio']
    return profile, annual_mean

def estimate_future_level(annual_mean):
    recent = annual_mean.iloc[-3:]
    years = recent.index.values
    vals = recent.values
    slope = np.polyfit(years - years.mean(), vals, 1)[0]
    base = float(annual_mean.iloc[-1])
    slope = np.clip(slope, -base * 0.05, base * 0.15)
    return base, slope

def predict_seasonal(sub_df, profile, base, slope):
    preds = []
    for _, row in sub_df.iterrows():
        dt = row['Date']
        level = base + slope * (dt.year - 2022)
        mask = (profile['month'] == dt.month) & (profile['dom'] == dt.day)
        ratio_rows = profile[mask]
        if not ratio_rows.empty:
            ratio = float(ratio_rows['seasonal_ratio'].values[0])
        else:
            m_mask = profile['month'] == dt.month
            ratio = float(profile[m_mask]['seasonal_ratio'].mean()) if m_mask.any() else 1.0
        preds.append(max(0, level * ratio))
    return np.array(preds)

## 3. LightGBM Modeling

In [ ]:
def train_lgb_detrended(df, target):
    df2 = df.copy()
    df2['year'] = df2['Date'].dt.year
    annual = df2.groupby('year')[target].mean()
    df2['norm'] = df2[target] / df2['year'].map(annual)
    X = calendar_features(df2['Date'])
    
    params = {
        'objective': 'regression_l1', 'metric': 'mae', 'learning_rate': 0.03,
        'num_leaves': 63, 'subsample': 0.8, 'colsample_bytree': 0.7,
        'min_child_samples': 15, 'reg_alpha': 0.1, 'reg_lambda': 0.1,
        'n_jobs': -1, 'random_state': 42, 'verbosity': -1
    }
    
    ds = lgb.Dataset(X, label=df2['norm'])
    model = lgb.train(params, ds, num_boost_round=3000)
    return model, annual

def predict_lgb(model, sub_df, base, slope):
    X_test = calendar_features(sub_df['Date'])
    preds_norm = model.predict(X_test)
    scales = np.array([base + slope * (yr - 2022) for yr in sub_df['Date'].dt.year])
    return np.clip(preds_norm * scales, 0, None)

## 4. Backtesting (Validation 2022)

In [ ]:
def backtest_strategy(target_name):
    cutoff = '2022-01-01'
    train_bt = df[df['Date'] < cutoff]
    val_bt = df[df['Date'] >= cutoff]
    
    # Seasonal
    prof, ann = build_seasonal_profile(train_bt, target_name)
    b, s = estimate_future_level(ann)
    p_s = predict_seasonal(val_bt, prof, b, s)
    mae_s = mean_absolute_error(val_bt[target_name], p_s)
    
    # LGB
    model, ann = train_lgb_detrended(train_bt, target_name)
    b, s = estimate_future_level(ann)
    p_l = predict_lgb(model, val_bt, b, s)
    mae_l = mean_absolute_error(val_bt[target_name], p_l)
    
    w_s = 1/mae_s / (1/mae_s + 1/mae_l)
    w_l = 1 - w_s
    return w_s, w_l, mae_s, mae_l

w_s_rev, w_l_rev, mae_s_rev, mae_l_rev = backtest_strategy('Revenue')
w_s_cog, w_l_cog, mae_s_cog, mae_l_cog = backtest_strategy('COGS')
print(f'Revenue Weights: Seasonal={w_s_rev:.2f}, LGB={w_l_rev:.2f}')

## 5. Final Production Forecast

In [ ]:
def get_final_forecast(target_name, w_s, w_l):
    prof, ann = build_seasonal_profile(df, target_name)
    base, slope = estimate_future_level(ann)
    p_s = predict_seasonal(sub, prof, base, slope)
    
    model, ann = train_lgb_detrended(df, target_name)
    base, slope = estimate_future_level(ann)
    p_l = predict_lgb(model, sub, base, slope)
    
    return w_s * p_s + w_l * p_l

rev_final = get_final_forecast('Revenue', w_s_rev, w_l_rev)
cog_final = get_final_forecast('COGS', w_s_cog, w_l_cog)

# Submission
submission = sub[['Date']].copy()
submission['Revenue'] = np.round(rev_final, 2)
submission['COGS'] = np.round(cog_final, 2)
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission.to_csv('../submission_v9_plus.csv', index=False)
print('Submission exported.')

## 6. Visualization

In [ ]:
plt.figure(figsize=(16, 6))
hist = df[df['Date'] >= '2022-01-01']
plt.plot(hist['Date'], hist['Revenue'], label='Actual 2022')
plt.plot(pd.to_datetime(submission['Date']), submission['Revenue'], label='Forecast 2023-24', ls='--')
plt.legend()
plt.show()